# 04 — Connected pipeline + dojo scoring

`sandbox eval pipeline --mock` scores **class, stage, extraction, and
routing** together. Without vendored mailroom the runner uses an offline
fallback (fake client / gold copy) and still emits the same score keys.

Scoring is [`llm-dojo-scoring` @ v0.11.0](https://github.com/Exios66/llm-dojo-scoring)
via [`eval/scoring.py`](../src/mailroom_sandbox/eval/scoring.py). Extraction
is **typed** (id/date/money/name…) — never exact-match-on-extraction.

v0.11.0 formulas and T0 names match v0.10.0; the release adds scoring docs
and `llm_dojo_scoring.prompts`.


In [ ]:
import sys
from pathlib import Path

def find_repo_root() -> Path:
    """Walk up from cwd (hostile kernels start in notebooks/)."""
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "pyproject.toml").is_file() and (cand / "reports").is_dir():
            return cand
    raise RuntimeError("repo root not found")

ROOT = find_repo_root()
sys.path.insert(0, str(ROOT / "notebooks"))
sys.path.insert(0, str(ROOT / "src"))

from _lib import bootstrap, isolate_outputs

ROOT = bootstrap(ROOT)
OUT = isolate_outputs(ROOT)
print("repo root :", ROOT)
print("sys.path[0]:", sys.path[0])
print("notebook outputs ->", OUT)


## Connected pipeline mock


In [ ]:
import llm_dojo_scoring as dojo
from mailroom_sandbox.eval.runners import (
    run_pipeline_eval,
    run_extract_eval,
    run_chained_eval,
    run_legalbench_eval,
)

print("dojo", getattr(dojo, "__version__", "pinned"))
pipe = run_pipeline_eval(mock=True, sample=3, connected=True, experiment_name="nb04_pipeline")
print("connected:", pipe.get("connected"))
print("scores   :", pipe["scores"])
assert pipe["scores"]["class_correct"] == 1.0
assert "stage_correct" in pipe["scores"]
assert "routing_accuracy" in pipe["scores"]


## Extract, chained sorter→extract, LegalBench Yes/No


In [ ]:
extract = run_extract_eval(mock=True, sample=3, experiment_name="nb04_extract")
chained = run_chained_eval(mock=True, sample=3, experiment_name="nb04_chained")
legal = run_legalbench_eval(mock=True, experiment_name="nb04_legalbench")
print("extract :", extract["scores"])
print("chained :", chained["scores"])
print("legal   :", legal["scores"])
print("chained composite is 0.25*sorter + 0.75*extract (see runners.run_chained_eval)")


## Direct dojo calls (classification + typed extraction)


In [ ]:
from mailroom_sandbox.datasets import load_manifest, parse_expected_fields
from mailroom_sandbox.eval import scoring

rows = load_manifest()
expected = [r["expected_doc_class"] for r in rows[:6]]
# Mock "model" copies gold — machinery proof.
cls = scoring.score_classification(expected, expected)
print("classification:", {k: cls[k] for k in ("exact_match", "n") if k in cls})
print("task keys     :", sorted((cls.get("task") or {}).keys())[:12])

msa = next(r for r in rows if r["id"] == "contract_msa")
gold = parse_expected_fields(msa) or {}
ext = scoring.score_extraction_row("contract", gold, gold, doc_text=None)
print("extraction overall:", ext.get("overall_extraction_score"), "doc_type=", ext.get("doc_type"))
print("HONEST GAP: scoring gold-against-gold is 1.0 by construction.")
print("Real extraction quality needs --local plus a specialist that can miss fields.")


## Prompt catalog (v0.11.0 additive surface)


In [ ]:
try:
    from llm_dojo_scoring.prompts import get_prompt, list_prompts
    names = list_prompts()
    print("catalog size:", len(names))
    intake = get_prompt("intake")
    print("intake.kind (must be deterministic, empty text):", intake.kind, repr(intake.text[:20] if intake.text else ""))
    sorter = get_prompt("sorter")
    print("sorter.kind:", sorter.kind, "text chars:", len(sorter.text or ""))
    print("Sandbox overlay templates still live in config/prompts/*_local_v0.txt")
    print("and win for local 7B/8B JSON-strict runs (`--prompt sorter_local_v0`).")
except Exception as exc:
    print("prompts catalog unavailable:", type(exc).__name__, exc)
    print("HONEST GAP: install llm-dojo-scoring @ v0.11.0 to import llm_dojo_scoring.prompts")
